# 01 — Data Preprocessing

This notebook performs the reproducible preprocessing of the raw
experimental datasets.

## Objectives

1. Load the selected raw datasets.
2. Preserve the original raw data without modification.
3. Standardize variable names and data types.
4. Validate participant identifiers and experimental conditions.
5. Identify missing, duplicated, and invalid observations.
6. Prepare questionnaire/SAM data.
7. Prepare trial-level n-back data.
8. Integrate participant-level information when appropriate.
9. Generate reproducible interim datasets.

Raw datasets are treated as read-only.

In [2]:
import pandas as pd
import numpy as np

from src import config as cfg

from src.utils import (
    initialize_project,
    dataframe_summary,
)

from src.preprocessing import (
    standardize_column_names,
    normalize_missing_values,
    convert_to_numeric,
    filter_reaction_times,
)

initialize_project()

In [3]:
raw_files = {
    "sam": cfg.RAW_SAM_FILE,
    "nback2": cfg.RAW_NBACK2_FILE,
    "nback4": cfg.RAW_NBACK4_FILE,
}

for name, path in raw_files.items():
    print(
        f"{name:<10} "
        f"{'OK' if path.exists() else 'MISSING'} "
        f"{path.name}"
    )

sam        OK DBs_SAMs_V9_200626.csv
nback2     OK DBS_N_back_2_01072026.sav
nback4     OK n_back_4.xlsx


In [4]:
sam_raw = pd.read_csv(
    cfg.RAW_SAM_FILE
)

nback2_raw = pd.read_spss(
    cfg.RAW_NBACK2_FILE
)

nback4_raw = pd.read_excel(
    cfg.RAW_NBACK4_FILE
)

*_raw
   ↓
imutável

cópia
   ↓
preprocessing

In [5]:
sam = sam_raw.copy()

nback2 = nback2_raw.copy()

nback4 = nback4_raw.copy()

In [6]:
datasets = {
    "SAM": sam_raw,
    "N-back 2": nback2_raw,
    "N-back 4": nback4_raw,
}

initial_audit = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "missing_cells": int(
                df.isna().sum().sum()
            ),
            "duplicated_rows": int(
                df.duplicated().sum()
            ),
        }
        for name, df in datasets.items()
    ]
)

initial_audit

,dataset,rows,columns,missing_cells,duplicated_rows
0,SAM,24,35,0,0
1,N-back 2,17896,17,775,56
2,N-back 4,17839,16,17947,0


In [7]:
sam = standardize_column_names(
    sam_raw.copy()
)

nback2 = standardize_column_names(
    nback2_raw.copy()
)

nback4 = standardize_column_names(
    nback4_raw.copy()
)

Participant_Name
        ↓
participant_name

SAM_Neg_V1_mean
        ↓
sam_neg_v1_mean

R_Reaction_Time
        ↓
r_reaction_time

In [8]:
print(sam.columns.tolist())

['participant_name', 'order', 'group_id', 'sex', 'age', 'cog_reap', 'ex_sup', 'sam_neg_v1_mean', 'sam_neg_v2_mean', 'sam_neg_v3_mean', 'sam_neg_v3_1_mean', 'sam_neg_v3_2_mean', 'sam_neg_v4_mean', 'sam_neg_v5_mean', 'sam_neg_v6_mean', 'sam_neg_v6_3_mean', 'sam_neg_v6_4_mean', 'sam_neg_v7_mean', 'sam_pos_v1_mean', 'sam_pos_v2_mean', 'sam_pos_v3_mean', 'sam_pos_v3_1_mean', 'sam_pos_v3_2_mean', 'sam_pos_v4_mean', 'sam_pos_v5_mean', 'sam_pos_v6_mean', 'sam_pos_v6_3_mean', 'sam_pos_v6_4_mean', 'sam_pos_v7_mean', 'traite_result_neg', 'traite_result_pos', 'result_state_neg_post', 'result_state_pos_post', 'result_state_neg_pre', 'result_state_pos_pre']


In [9]:
print(nback4.columns.tolist())

['order', 'participant_group', 'participant_name', 'session_id', 'block_name', 'trial_name', 'event_name', 'cumulative_time', 'participant_response', 'key', 'pressed_or_released', 'correct_response', 'r_reaction_time', 'error_code', 'all_loction_trial_variable', 'cr']


Original variable: Participant_Name

Observed values: numeric identifiers

Proposed standardized variable: participant_id

Status: pending confirmation

In [10]:
COLUMN_RENAME_MAP = {
    "participant_name": "participant_id",
}

In [ ]:
nback4["cr"].value_counts(
    dropna=False
)

cr
Correct        15084
Incorrect       2720
Not Respond       18
NaN               16
                   1
Name: count, dtype: int64

In [13]:
nback4["error_code"].value_counts(
    dropna=False
)

error_code
C      17795
NR        19
E         18
SC         4
NaN        3
Name: count, dtype: int64

In [14]:
nback4[
    "block_name"
].value_counts(
    dropna=False
)

block_name
n_back2    5958
n_back3    5941
n_back1    5940
Name: count, dtype: int64

In [15]:
nback4[
    "session_id"
].value_counts(
    dropna=False
)

session_id
1st    9315
2nd    8524
Name: count, dtype: int64

In [18]:
rt = pd.to_numeric(
    nback4["r_reaction_time"],
    errors="coerce",
)

rt.describe(
    percentiles=[
        .01,
        .05,
        .25,
        .50,
        .75,
        .95,
        .99,
    ]
)

count     810.000000
mean      661.025926
std       659.452438
min        15.000000
1%         31.360000
5%        123.450000
25%       277.000000
50%       423.500000
75%       810.000000
95%      1903.950000
99%      3166.820000
max      5789.000000
Name: r_reaction_time, dtype: float64

In [21]:
from src.config import (
    RT_MIN_MS,
    RT_MAX_MS,
)

print(
    "RT < 100 ms:",
    (rt < RT_MIN_MS).sum()
)

print(
    "RT > 5000 ms:",
    (rt > RT_MAX_MS).sum()
)

RT < 100 ms: 31
RT > 5000 ms: 1


SAM raw
   ↓
SAM preprocessing
   ↓
sam_long


N-back 2 raw
   ↓
N-back preprocessing
   ↓
nback2_trials


N-back 4 raw
   ↓
N-back preprocessing
   ↓
nback4_trials

In [26]:
nback4.head(5)

,order,participant_group,participant_name,session_id,block_name,trial_name,event_name,cumulative_time,participant_response,key,pressed_or_released,correct_response,r_reaction_time,error_code,all_loction_trial_variable,cr
0,27,B,1101,1st,n_back1,"LOP_n=1, 1",Q,164467,Z,Z,Pressed,/; Z,425,C,NaN,Correct
1,30,B,1101,1st,n_back1,"LOP_n=1, 2",Q,167452,Z,Z,Pressed,/; Z,1375,C,NaN,Correct
2,33,B,1101,1st,n_back1,"LOP_n=1, 3",Q,169590,Z,Z,Pressed,/; Z,523,C,NaN,Correct
3,36,B,1101,1st,n_back1,"LOP_n=1, 4",Q,171550,Z,Z,Pressed,/; Z,345,C,NaN,Correct
4,39,B,1101,1st,n_back1,"LOP_n=1, 5",Q,173487,Z,Z,Pressed,/; Z,327,C,NaN,Correct
